In [1]:
import re

In [2]:
import json

In [3]:
import tqdm

In [4]:
from dotenv import load_dotenv

In [5]:
load_dotenv()

True

In [6]:
import pandas as pd

In [7]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [8]:
MODEL_ID = "google/gemma-4-E2B-it"

In [9]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [10]:
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto")

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [11]:
df = pd.read_excel("data_raw.xlsx", header=0)

In [12]:
df

,title_rus,title_eng,goals,tasks,annotation,description,expectations,product_result,result_criterias,social_effect,commercial_effect
0,Исследование приоритетов и механизмов реализац...,Study of Priorities and Mechanisms for Impleme...,Целью проекта является эмпирическая проверка г...,1) Анализ повестки международных доноров\n2) А...,Работа международных фондов (доноров) должна п...,"Согласно определению международных фондов, про...",Аналитический отчет по избранным странам.\n\nС...,NaN,NaN,NaN,NaN
1,Антрополе - научно-популярный видео-подкаст о ...,Anthropole is a Popular Science Video Podcast ...,Выпустить и популяризовать 27 видео-подкастов ...,Снять и смонтировать подкасты;\nРазработать ст...,"\tАнтрополе - научно-популярный проект, в рамк...","Социальное знание близко и интересно обществу,...","Студенты получат опыт монтажа, продвижения и к...",Регулярный видео-подкаст (+экспедиционные филь...,Опубликованы 27 видео-подкастов о необычных со...,Популяризация социального знания должна привес...,Планируется в течение года выйти на окупаемост...
2,"Разработка, создание и ведение сайта, посвящен...","Design, Development and Implementation of a We...","Для того, чтобы получить более полное представ...",Подготовка технического задания для разработчи...,Художественное образование и творчество художн...,Тема обучения арабских художников в художестве...,"Навыки создания сайта (полный цикл, от подгото...","Создание сайта, посвященного истории арабских ...",Выполнение заданий руководителя проекта,Укрепление международных связей с художниками ...,NaN
3,Перевод с английского языка коллективной моног...,Translation from English of the collective mon...,Результатом проекта станет качественный перево...,Результатом проекта станет качественный перево...,"Коллективная монография, авторы которой являют...","Коллективная монография, авторы которой являют...",Участники проекта приобретут новые знания в об...,Результатом проекта станет качественный перево...,Критериями достижения результата будет возможн...,NaN,NaN
4,Сеть военно-политических союзов в Евразии: баз...,Network of Military in Eurasia: a Database,Целью проекта является создание базы данных со...,1.\tОпределение методики включения союзов в ба...,Проект посвящен изучению сети военно-политичес...,Проект посвящен анализу истории существования ...,1.\tОбучение навыкам сбора и анализа данных 2....,База данных союзов 1815-2024 в Евразии.,Создание базы данных как минимум тысячи диадны...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1253,Влияние мер контроля за движением капитала на ...,The Impact of Capital Controls on the Investme...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1254,"Создание страницы (лендинга) проектов ПУЛ ""Раз...",Creation of a Landing Page for Projects of the...,Основная цель: создать эффективный digital-инс...,Разработать информационную архитектуру и прото...,Проект направлен на разработку и запуск одност...,Проектно-учебная лаборатория «Развитие универс...,Навыки анализа потребностей заказчика.\nОпыт р...,"Рабочий, опубликованный в интернете лендинг.",Функциональные: Все разделы лендинга работают ...,NaN,"Повышение конверсии запросов в контракты, форм..."
1255,Разработка стратегической и контентно-коммуник...,Development of a Strategic and Content-based C...,1. Разработать целостную стратегию присутствия...,Для маркетолога (стратега):\n- провести аудит ...,Проект предполагает разработку системной комму...,Экосистема «Вулканариум–Ойкумена» объединяет н...,Участники:\n- овладеют навыком комплексного ау...,Стратегический документ (дорожная карта на 3–6...,- Наличие полноценного стратегического докумен...,- поддержка просветительской миссии музея «Вул...,- повышение эффективности продвижения услуг (э...
1256,Актив Центра развития карьеры - профориентацио...,Active Club of the Career Development Center -...,Сформировать у студентов Питерской Вышки базов...,1. Разработать и реализовать мероприятия Центр...,Актив Центра развития карьеры — это проектная ...,Актуальность про

In [13]:
df = df.fillna("")

In [14]:
df[df.duplicated(subset=["title_rus"])]["title_rus"].nunique()

31

In [15]:
df.shape

(1258, 11)

In [16]:
df = df.drop_duplicates(subset=["title_rus"], inplace=False).reset_index(drop=True, inplace=False)

In [17]:
df.shape

(1220, 11)

In [18]:
df

,title_rus,title_eng,goals,tasks,annotation,description,expectations,product_result,result_criterias,social_effect,commercial_effect
0,Исследование приоритетов и механизмов реализац...,Study of Priorities and Mechanisms for Impleme...,Целью проекта является эмпирическая проверка г...,1) Анализ повестки международных доноров\n2) А...,Работа международных фондов (доноров) должна п...,"Согласно определению международных фондов, про...",Аналитический отчет по избранным странам.\n\nС...,,,,
1,Антрополе - научно-популярный видео-подкаст о ...,Anthropole is a Popular Science Video Podcast ...,Выпустить и популяризовать 27 видео-подкастов ...,Снять и смонтировать подкасты;\nРазработать ст...,"\tАнтрополе - научно-популярный проект, в рамк...","Социальное знание близко и интересно обществу,...","Студенты получат опыт монтажа, продвижения и к...",Регулярный видео-подкаст (+экспедиционные филь...,Опубликованы 27 видео-подкастов о необычных со...,Популяризация социального знания должна привес...,Планируется в течение года выйти на окупаемост...
2,"Разработка, создание и ведение сайта, посвящен...","Design, Development and Implementation of a We...","Для того, чтобы получить более полное представ...",Подготовка технического задания для разработчи...,Художественное образование и творчество художн...,Тема обучения арабских художников в художестве...,"Навыки создания сайта (полный цикл, от подгото...","Создание сайта, посвященного истории арабских ...",Выполнение заданий руководителя проекта,Укрепление международных связей с художниками ...,
3,Перевод с английского языка коллективной моног...,Translation from English of the collective mon...,Результатом проекта станет качественный перево...,Результатом проекта станет качественный перево...,"Коллективная монография, авторы которой являют...","Коллективная монография, авторы которой являют...",Участники проекта приобретут новые знания в об...,Результатом проекта станет качественный перево...,Критериями достижения результата будет возможн...,,
4,Сеть военно-политических союзов в Евразии: баз...,Network of Military in Eurasia: a Database,Целью проекта является создание базы данных со...,1.\tОпределение методики включения союзов в ба...,Проект посвящен изучению сети военно-политичес...,Проект посвящен анализу истории существования ...,1.\tОбучение навыкам сбора и анализа данных 2....,База данных союзов 1815-2024 в Евразии.,Создание базы данных как минимум тысячи диадны...,,
...,...,...,...,...,...,...,...,...,...,...,...
1215,Влияние мер контроля за движением капитала на ...,The Impact of Capital Controls on the Investme...,,,,,,,,,
1216,"Создание страницы (лендинга) проектов ПУЛ ""Раз...",Creation of a Landing Page for Projects of the...,Основная цель: создать эффективный digital-инс...,Разработать информационную архитектуру и прото...,Проект направлен на разработку и запуск одност...,Проектно-учебная лаборатория «Развитие универс...,Навыки анализа потребностей заказчика.\nОпыт р...,"Рабочий, опубликованный в интернете лендинг.",Функциональные: Все разделы лендинга работают ...,,"Повышение конверсии запросов в контракты, форм..."
1217,Разработка стратегической и контентно-коммуник...,Development of a Strategic and Content-based C...,1. Разработать целостную стратегию присутствия...,Для маркетолога (стратега):\n- провести аудит ...,Проект предполагает разработку системной комму...,Экосистема «Вулканариум–Ойкумена» объединяет н...,Участники:\n- овладеют навыком комплексного ау...,Стратегический документ (дорожная карта на 3–6...,- Наличие полноценного стратегического докумен...,- поддержка просветительской миссии музея «Вул...,- повышение эффективности продвижения услуг (э...
1218,Актив Центра развития карьеры - профориентацио...,Active Club of the Career Development Center -...,Сформировать у студентов Питерской Вышки базов...,1. Разработать и реализовать мероприятия Центр...,Актив Центра развития карьеры — это проектная ...,Актуальность проекта «Актив Центра развития ка...,1. Навыки самостоятельн

In [19]:
df[df.duplicated(subset=["title_rus"])]

,title_rus,title_eng,goals,tasks,annotation,description,expectations,product_result,result_criterias,social_effect,commercial_effect


In [20]:
SYSTEM_PROMPT = """
You are a helpful assistant, which can validate whether or not a provided student project title is a valid project title in russian.
In case title is valid, return 1. Otherwise 0.
Return strictly JSON as in example below.
```json
{"is_valid": 1}
```
"""

In [21]:
messages = []

In [22]:
for row in df.iterrows():
    messages.append(
        (
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row[1]["title_rus"]},
        )
    )

In [23]:
idx = []

In [24]:
for message in tqdm.tqdm(messages):
    text = processor.apply_chat_template(
        message, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    outputs = model.generate(**inputs, max_new_tokens=1024)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    
    json_match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
    if json_match:
        json_str = json_match.group(1)
    else:
        json_str = response
    
    try:
        answer = json.loads(json_str)
        idx.append(answer.get("is_valid", 0))
    except Exception:
        idx.append(0)


100%|██████████| 1220/1220 [26:52<00:00,  1.32s/it]


In [25]:
idx = [True if x else False for x in idx]

In [26]:
len(df)

1220

In [27]:
df = df[idx].reset_index(drop=True)

In [28]:
len(df)

1187

In [29]:
df[["title_rus", "title_eng", "annotation", "description"]].to_excel("data_clean.xlsx", index=False, header=True)